# Chapter 9: Reward Modeling & The Critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part2_core/09_reward_modeling.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar & Michael Chertushkin (Packt, 2025)  
> **Chapter 9:** Reward Modeling & The Critic  
> **Notebook:** `part2_core/09_reward_modeling.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 9 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q transformers==4.40.0 datasets==2.19.0 accelerate==0.29.3

print('Environment ready.')

In [ ]:
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Part 1 — Reward Model Architecture

### Why a classification backbone?

A reward model needs to produce a **single scalar** per input sequence. We choose `Qwen/Qwen2.5-0.5B` (66 M parameters) as the backbone because:
- Its encoder architecture naturally produces a `[CLS]` pooled representation for the whole sequence.
- It is bidirectional — it can attend to the full `(prompt, response)` jointly.
- At 66 M parameters it fits easily on a T4 with room to spare.

```
Input: "[CLS] {prompt} [SEP] {response} [SEP]"
         ↓
    distilbert encoder (66M)
         ↓
  [CLS] token hidden state  (768-d)
         ↓
    Linear(768 → 1)  +  Dropout
         ↓
    scalar reward  r ∈ ℝ
```

In [ ]:
BACKBONE = 'Qwen/Qwen2.5-0.5B'


class RewardModel(nn.Module):
    """
    distilbert encoder + scalar linear head.
    Takes (prompt, response) text and returns a scalar reward.
    """

    def __init__(self, backbone_name: str = BACKBONE, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone_name)
        hidden = self.encoder.config.hidden_size   # 768
        self.dropout = nn.Dropout(dropout)
        self.reward_head = nn.Linear(hidden, 1)
        nn.init.normal_(self.reward_head.weight, std=0.02)
        nn.init.zeros_(self.reward_head.bias)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_hidden = out.last_hidden_state[:, 0, :]   # [CLS] token  (B, H)
        cls_hidden = self.dropout(cls_hidden)
        return self.reward_head(cls_hidden).squeeze(-1)  # (B,)


rm_tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
reward_model = RewardModel(BACKBONE).to(DEVICE)

n_params = sum(p.numel() for p in reward_model.parameters())
print(f'Reward model parameters: {n_params / 1e6:.1f} M')
print(f'Backbone hidden size   : {reward_model.encoder.config.hidden_size}')

## Part 2 — Preference Dataset (30 pairs)

Each preference pair contains:
- `prompt`   — a user question
- `chosen`   — a helpful, informative response
- `rejected` — an unhelpful or vague response

A real deployment would source these from trained human annotators.

In [ ]:
PREFERENCE_PAIRS = [
    {"prompt": "What is the capital of France?",
     "chosen":   "The capital of France is Paris, famous for the Eiffel Tower and the Louvre museum.",
     "rejected": "France has a capital city."},
    {"prompt": "Explain photosynthesis.",
     "chosen":   "Photosynthesis is the process by which plants use sunlight, CO2, and water to produce glucose and oxygen.",
     "rejected": "Plants get energy from the sun."},
    {"prompt": "What is machine learning?",
     "chosen":   "Machine learning is a branch of AI where algorithms learn patterns from data rather than following explicit rules.",
     "rejected": "Computers learn things."},
    {"prompt": "What is 15 times 7?",
     "chosen":   "15 times 7 equals 105.",
     "rejected": "A big number."},
    {"prompt": "How do vaccines work?",
     "chosen":   "Vaccines expose the immune system to a harmless antigen, training it to fight the real pathogen if encountered later.",
     "rejected": "They stop you getting sick."},
    {"prompt": "What is DNA?",
     "chosen":   "DNA (deoxyribonucleic acid) is the double-helix molecule that carries genetic information in all living organisms.",
     "rejected": "DNA is in your body."},
    {"prompt": "Who wrote Romeo and Juliet?",
     "chosen":   "Romeo and Juliet was written by William Shakespeare, probably around 1594-1596.",
     "rejected": "Some famous writer."},
    {"prompt": "What is Newton's second law?",
     "chosen":   "Newton's second law states that force equals mass times acceleration: F = ma.",
     "rejected": "Objects have forces."},
    {"prompt": "What is the boiling point of water?",
     "chosen":   "Water boils at 100 degrees Celsius (212 Fahrenheit) at standard atmospheric pressure.",
     "rejected": "Water gets very hot."},
    {"prompt": "What does GPU stand for?",
     "chosen":   "GPU stands for Graphics Processing Unit, a processor optimised for parallel computation.",
     "rejected": "A computer chip."},
    {"prompt": "Explain the water cycle.",
     "chosen":   "Water evaporates from oceans, rises and condenses into clouds, and falls as precipitation to flow back to water bodies.",
     "rejected": "Water goes up and comes back down."},
    {"prompt": "What is the speed of light?",
     "chosen":   "The speed of light in a vacuum is approximately 299,792,458 metres per second.",
     "rejected": "Light is very fast."},
    {"prompt": "Translate hello into French.",
     "chosen":   "Hello in French is Bonjour.",
     "rejected": "French uses different words."},
    {"prompt": "What is the square root of 256?",
     "chosen":   "The square root of 256 is 16.",
     "rejected": "Around 15 or so."},
    {"prompt": "What is climate change?",
     "chosen":   "Climate change refers to long-term shifts in global temperatures and weather patterns, primarily driven by human emissions of greenhouse gases.",
     "rejected": "The weather is changing."},
    {"prompt": "What is the largest planet?",
     "chosen":   "Jupiter is the largest planet in the solar system, with a mass more than twice that of all other planets combined.",
     "rejected": "A big planet."},
    {"prompt": "Who invented the light bulb?",
     "chosen":   "Thomas Edison is credited with developing a practical incandescent light bulb in 1879, though others made earlier contributions.",
     "rejected": "Some inventor."},
    {"prompt": "What is Python used for?",
     "chosen":   "Python is widely used for data science, machine learning, web development, automation, and scientific computing.",
     "rejected": "Programming stuff."},
    {"prompt": "How does the heart work?",
     "chosen":   "The heart is a four-chambered pump that pushes oxygenated blood through the body via rhythmic contractions.",
     "rejected": "It pumps blood around."},
    {"prompt": "What is the Pythagorean theorem?",
     "chosen":   "The Pythagorean theorem states that in a right triangle a^2 + b^2 = c^2, where c is the hypotenuse.",
     "rejected": "A math formula."},
    {"prompt": "What is gravity?",
     "chosen":   "Gravity is the fundamental force of attraction between masses; it keeps planets in orbit and objects on Earth's surface.",
     "rejected": "What makes things fall."},
    {"prompt": "What is the internet?",
     "chosen":   "The internet is a global network of interconnected computers that share data using standardised protocols like TCP/IP.",
     "rejected": "A network of computers."},
    {"prompt": "What is the atomic number of carbon?",
     "chosen":   "The atomic number of carbon is 6, meaning it has 6 protons in its nucleus.",
     "rejected": "Six-ish."},
    {"prompt": "What is blockchain?",
     "chosen":   "Blockchain is a distributed ledger technology where records are stored in cryptographically linked blocks shared across a network.",
     "rejected": "A crypto thing."},
    {"prompt": "What does USB stand for?",
     "chosen":   "USB stands for Universal Serial Bus, a standard interface for connecting devices to computers.",
     "rejected": "A plug-in connector."},
    {"prompt": "What is the capital of Australia?",
     "chosen":   "The capital of Australia is Canberra, which was purpose-built as a compromise between Sydney and Melbourne.",
     "rejected": "Sydney maybe?"},
    {"prompt": "What is inflation?",
     "chosen":   "Inflation is the rate at which the general price level of goods and services rises, reducing the purchasing power of money.",
     "rejected": "Prices go up."},
    {"prompt": "How many bones are in the human body?",
     "chosen":   "An adult human body has 206 bones. Babies are born with around 270, which fuse as they grow.",
     "rejected": "Lots of bones."},
    {"prompt": "What is the Big Bang theory?",
     "chosen":   "The Big Bang theory is the cosmological model describing the universe's origin as an extremely hot, dense state about 13.8 billion years ago.",
     "rejected": "The universe started somehow."},
    {"prompt": "What is a neuron?",
     "chosen":   "A neuron is a specialised nerve cell that transmits electrical and chemical signals throughout the nervous system.",
     "rejected": "A brain cell."},
]

# Split: 25 train, 5 validation
random.shuffle(PREFERENCE_PAIRS)
TRAIN_PAIRS = PREFERENCE_PAIRS[:25]
VAL_PAIRS   = PREFERENCE_PAIRS[25:]
print(f'Training pairs   : {len(TRAIN_PAIRS)}')
print(f'Validation pairs : {len(VAL_PAIRS)}')

## Part 3 — Bradley-Terry Loss from Scratch

Given a chosen response $y_w$ and a rejected response $y_l$ to the same prompt $x$, the Bradley-Terry model asserts:

$$P(y_w \succ y_l \mid x) = \sigma\bigl(r_\theta(x, y_w) - r_\theta(x, y_l)\bigr)$$

We minimise the negative log-likelihood:

$$\mathcal{L}_{\text{RM}} = -\log\,\sigma\bigl(r_\theta(x, y_w) - r_\theta(x, y_l)\bigr)$$

Geometrically: we want $r(y_w) - r(y_l)$ to be large and positive.

In [ ]:
def bradley_terry_loss(
    r_chosen: torch.Tensor,
    r_rejected: torch.Tensor,
) -> torch.Tensor:
    """
    L = -log sigmoid(r_w - r_l)

    Parameters
    ----------
    r_chosen   : (B,) reward scores for chosen responses
    r_rejected : (B,) reward scores for rejected responses

    Returns
    -------
    scalar loss
    """
    margin = r_chosen - r_rejected          # positive = model prefers chosen
    return -F.logsigmoid(margin).mean()     # minimised when margin >> 0


# Demonstrate the loss surface
margins = torch.linspace(-3, 3, 200)
losses  = -F.logsigmoid(margins).numpy()

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(margins.numpy(), losses, linewidth=2, color='steelblue')
ax.axvline(0, color='r', linestyle='--', alpha=0.6, label='margin = 0 (random)')
ax.set_xlabel('Reward margin r(y_w) - r(y_l)')
ax.set_ylabel('Bradley-Terry Loss')
ax.set_title('Loss decreases as margin grows positive')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('bt_loss_curve.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def tokenize_rm(
    prompt: str,
    response: str,
    tokenizer,
    max_len: int = 128,
):
    """
    Encode (prompt, response) for the reward model.
    distilbert handles [CLS] prompt [SEP] response [SEP] automatically.
    """
    enc = tokenizer(
        prompt, response,
        return_tensors='pt',
        truncation=True,
        max_length=max_len,
        padding='max_length',
    )
    return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)


# Verify shape
ids, mask = tokenize_rm('Test prompt', 'Test response', rm_tokenizer)
print(f'input_ids shape     : {ids.shape}')
print(f'attention_mask shape: {mask.shape}')

## Part 4 — Training the Reward Model

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

rm_optimizer = AdamW(reward_model.parameters(), lr=2e-5, weight_decay=0.01)
NUM_EPOCHS = 5
scheduler  = CosineAnnealingLR(rm_optimizer, T_max=NUM_EPOCHS)

train_losses, train_accs = [], []
val_losses,   val_accs   = [], []


def evaluate(model, tokenizer, pairs, device):
    model.eval()
    losses, correct = [], 0
    with torch.no_grad():
        for pair in pairs:
            ids_w, mask_w = tokenize_rm(pair['prompt'], pair['chosen'],   tokenizer)
            ids_l, mask_l = tokenize_rm(pair['prompt'], pair['rejected'], tokenizer)
            ids_w  = ids_w.unsqueeze(0).to(device)
            mask_w = mask_w.unsqueeze(0).to(device)
            ids_l  = ids_l.unsqueeze(0).to(device)
            mask_l = mask_l.unsqueeze(0).to(device)
            r_w = model(ids_w, mask_w)
            r_l = model(ids_l, mask_l)
            loss = bradley_terry_loss(r_w, r_l)
            losses.append(loss.item())
            if r_w.item() > r_l.item():
                correct += 1
    return float(np.mean(losses)), correct / len(pairs)


print('Training reward model ...')
for epoch in range(NUM_EPOCHS):
    reward_model.train()
    random.shuffle(TRAIN_PAIRS)
    epoch_losses, correct = [], 0

    for pair in TRAIN_PAIRS:
        ids_w, mask_w = tokenize_rm(pair['prompt'], pair['chosen'],   rm_tokenizer)
        ids_l, mask_l = tokenize_rm(pair['prompt'], pair['rejected'], rm_tokenizer)
        ids_w  = ids_w.unsqueeze(0).to(DEVICE)
        mask_w = mask_w.unsqueeze(0).to(DEVICE)
        ids_l  = ids_l.unsqueeze(0).to(DEVICE)
        mask_l = mask_l.unsqueeze(0).to(DEVICE)

        r_w = reward_model(ids_w, mask_w)
        r_l = reward_model(ids_l, mask_l)
        loss = bradley_terry_loss(r_w, r_l)

        rm_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        rm_optimizer.step()

        epoch_losses.append(loss.item())
        if r_w.item() > r_l.item():
            correct += 1

    scheduler.step()

    t_loss = float(np.mean(epoch_losses))
    t_acc  = correct / len(TRAIN_PAIRS)
    v_loss, v_acc = evaluate(reward_model, rm_tokenizer, VAL_PAIRS, DEVICE)

    train_losses.append(t_loss)
    train_accs.append(t_acc)
    val_losses.append(v_loss)
    val_accs.append(v_acc)

    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
          f'train_loss={t_loss:.4f}  train_acc={t_acc:.2%}  '
          f'val_loss={v_loss:.4f}  val_acc={v_acc:.2%}')

print('Training complete!')

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses, 'b-o', linewidth=2, label='train')
ax1.plot(epochs, val_losses,   'r-s', linewidth=2, label='val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Bradley-Terry Loss')
ax1.set_title('Reward Model — Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_accs, 'b-o', linewidth=2, label='train')
ax2.plot(epochs, val_accs,   'r-s', linewidth=2, label='val')
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='random baseline')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Pairwise Accuracy')
ax2.set_title('Reward Model — Pairwise Accuracy')
ax2.set_ylim([0, 1.05])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rm_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 5 — Calibration Analysis

A well-calibrated reward model should assign **higher scores to objectively better responses**. We test this on 10 held-out examples with hand-assigned quality ratings from 1 (very bad) to 5 (excellent).

In [ ]:
CALIBRATION_EXAMPLES = [
    {"prompt": "What is 2+2?",
     "response": "2 plus 2 equals 4.",
     "human_rating": 5},
    {"prompt": "What is 2+2?",
     "response": "It's about 5 I think.",
     "human_rating": 1},
    {"prompt": "What is the capital of Japan?",
     "response": "The capital of Japan is Tokyo, a city of over 13 million people.",
     "human_rating": 5},
    {"prompt": "What is the capital of Japan?",
     "response": "Japan has cities.",
     "human_rating": 1},
    {"prompt": "Who wrote Hamlet?",
     "response": "Hamlet was written by William Shakespeare around 1600.",
     "human_rating": 5},
    {"prompt": "Who wrote Hamlet?",
     "response": "A famous playwright wrote it.",
     "human_rating": 2},
    {"prompt": "What is gravity?",
     "response": "Gravity is the force of attraction between masses that keeps planets in orbit.",
     "human_rating": 4},
    {"prompt": "What is gravity?",
     "response": "Gravity is why things fall down.",
     "human_rating": 3},
    {"prompt": "Explain machine learning briefly.",
     "response": "Machine learning is a field of AI where models learn patterns from data to make predictions without explicit programming.",
     "human_rating": 5},
    {"prompt": "Explain machine learning briefly.",
     "response": "It's computers doing things automatically.",
     "human_rating": 2},
]

reward_model.eval()
model_scores = []
human_scores = []

for ex in CALIBRATION_EXAMPLES:
    ids, mask = tokenize_rm(ex['prompt'], ex['response'], rm_tokenizer)
    ids  = ids.unsqueeze(0).to(DEVICE)
    mask = mask.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        r = reward_model(ids, mask).item()
    model_scores.append(r)
    human_scores.append(ex['human_rating'])
    print(f'  human={ex["human_rating"]}  model={r:+.4f}  "{ex["response"][:60]}"')

rho, p_value = spearmanr(human_scores, model_scores)
print(f'\nSpearman correlation: ρ={rho:.3f}  (p={p_value:.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(human_scores, model_scores, c=human_scores, cmap='RdYlGn',
                     s=120, zorder=3, edgecolors='black', linewidths=0.5)
plt.colorbar(scatter, ax=ax, label='Human Rating')

# Trend line
z = np.polyfit(human_scores, model_scores, 1)
p = np.poly1d(z)
xs = np.linspace(min(human_scores) - 0.2, max(human_scores) + 0.2, 50)
ax.plot(xs, p(xs), 'k--', alpha=0.4, label=f'trend (ρ={rho:.2f})')

ax.set_xlabel('Human Quality Rating (1-5)')
ax.set_ylabel('Reward Model Score')
ax.set_title('Reward Model Calibration')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rm_calibration.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 6 — Reward Hacking Demo

**Reward hacking** occurs when the policy finds responses that score highly on the reward model but are not actually high quality. This is one of the most dangerous failure modes in RLHF.

We demonstrate three classes of adversarial responses:
1. **Verbosity hacking** — pad the answer with repetitive words.
2. **Keyword stuffing** — use many domain-sounding words without substance.
3. **Length hacking** — repeat the question back as a statement.

The reward model, if poorly calibrated, may score these higher than they deserve.

In [ ]:
ADVERSARIAL_EXAMPLES = [
    {
        "prompt": "What is the capital of France?",
        "good_response": "The capital of France is Paris.",
        "adversarial": (
            "The capital of France is an important and significant city that is "
            "well known and frequently discussed and often mentioned and commonly "
            "referred to and widely acknowledged and broadly accepted. "
            " " * 20  # artificial padding
        ),
        "hack_type": "verbosity hack",
    },
    {
        "prompt": "What is machine learning?",
        "good_response": "Machine learning is a branch of AI where models learn patterns from data.",
        "adversarial": (
            "Machine learning neural network deep learning algorithm data science "
            "model training optimization gradient descent backpropagation loss function "
            "weights bias epoch batch regularization dropout convolutional recurrent "
            "transformer attention BERT GPT supervised unsupervised reinforcement."
        ),
        "hack_type": "keyword stuffing",
    },
    {
        "prompt": "What is the boiling point of water?",
        "good_response": "Water boils at 100 degrees Celsius at standard pressure.",
        "adversarial": (
            "The boiling point of water is indeed the boiling point of water which "
            "is related to the temperature at which water boils and reaches its "
            "boiling temperature which is when water transitions from liquid state."
        ),
        "hack_type": "circular repetition hack",
    },
]


def score(prompt, response):
    ids, mask = tokenize_rm(prompt, response.strip()[:400], rm_tokenizer)
    ids  = ids.unsqueeze(0).to(DEVICE)
    mask = mask.unsqueeze(0).to(DEVICE)
    reward_model.eval()
    with torch.no_grad():
        return reward_model(ids, mask).item()


print('=== Reward Hacking Demo ===')
hacking_results = []
for ex in ADVERSARIAL_EXAMPLES:
    s_good = score(ex['prompt'], ex['good_response'])
    s_adv  = score(ex['prompt'], ex['adversarial'])
    hacked = s_adv > s_good
    hacking_results.append({'good': s_good, 'adv': s_adv, 'hacked': hacked,
                             'type': ex['hack_type']})
    print(f'\nHack type: {ex["hack_type"]}')
    print(f'  Prompt          : {ex["prompt"]}')
    print(f'  Good response   : score={s_good:+.4f}')
    print(f'  Adversarial     : score={s_adv:+.4f}')
    print(f'  HACKED          : {hacked}')

In [ ]:
labels = [r['type'] for r in hacking_results]
goods  = [r['good'] for r in hacking_results]
advs   = [r['adv']  for r in hacking_results]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, goods, 0.4, label='Good response',   color='steelblue')
ax.bar(x + 0.2, advs,  0.4, label='Adversarial',     color='crimson',  alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Reward Score')
ax.set_title('Reward Hacking: Good vs. Adversarial Response Scores')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('reward_hacking.png', dpi=100, bbox_inches='tight')
plt.show()

n_hacked = sum(r['hacked'] for r in hacking_results)
print(f'Hacking succeeded on {n_hacked}/{len(hacking_results)} examples.')
print('Note: a well-trained RM on large data would be harder to hack.')

## Part 7 — Out-of-Distribution Generalisation

A reward model trained on factual Q&A pairs should generalise to similar questions it has never seen. We test on prompts from a different domain (basic science and geography) and verify that the reward ordering still makes intuitive sense.

In [ ]:
OOD_PAIRS = [
    {"prompt": "What is the tallest mountain on Earth?",
     "chosen":   "Mount Everest is the tallest mountain on Earth, standing at 8,848 metres above sea level.",
     "rejected": "A really tall mountain."},
    {"prompt": "How many moons does Mars have?",
     "chosen":   "Mars has two moons: Phobos and Deimos.",
     "rejected": "Mars has moons."},
    {"prompt": "What is the longest river?",
     "chosen":   "The Nile River in Africa is traditionally considered the longest river at approximately 6,650 kilometres.",
     "rejected": "A very long river somewhere."},
    {"prompt": "What is the periodic table?",
     "chosen":   "The periodic table is a tabular arrangement of chemical elements ordered by atomic number and electron configuration.",
     "rejected": "A table with elements."},
    {"prompt": "What is a black hole?",
     "chosen":   "A black hole is a region of spacetime with gravity so strong that nothing, not even light, can escape.",
     "rejected": "A very dark hole in space."},
]

print('=== OOD Generalisation Test ===')
ood_correct = 0
for pair in OOD_PAIRS:
    sc = score(pair['prompt'], pair['chosen'])
    sr = score(pair['prompt'], pair['rejected'])
    correct = sc > sr
    if correct:
        ood_correct += 1
    print(f'  {"OK" if correct else "FAIL"} | chosen={sc:+.4f} rejected={sr:+.4f} | {pair["prompt"]}')

print(f'\nOOD accuracy: {ood_correct}/{len(OOD_PAIRS)} = {ood_correct/len(OOD_PAIRS):.2%}')

In [ ]:
ood_chosen   = [score(p['prompt'], p['chosen'])   for p in OOD_PAIRS]
ood_rejected = [score(p['prompt'], p['rejected']) for p in OOD_PAIRS]

xi = np.arange(len(OOD_PAIRS))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(xi - 0.2, ood_chosen,   0.4, label='Chosen',   color='steelblue')
ax.bar(xi + 0.2, ood_rejected, 0.4, label='Rejected',  color='salmon')
ax.set_xticks(xi)
ax.set_xticklabels([f'OOD {i+1}' for i in range(len(OOD_PAIRS))])
ax.set_ylabel('Reward Score')
ax.set_title('OOD Generalisation: Chosen vs. Rejected Scores')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('ood_generalisation.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary — Reward Modeling & The Critic

| Component | Detail |
|-----------|--------|
| Backbone | Qwen/Qwen2.5-0.5B (66 M params) |
| Head | Linear(768 → 1) + Dropout(0.1) |
| Loss | Bradley-Terry: $-\log\sigma(r_w - r_l)$ |
| Training data | 25 preference pairs |
| Validation data | 5 preference pairs |
| Calibration | Spearman correlation with human ratings |

### Key lessons
- **Pairwise training** (Bradley-Terry) is more sample-efficient than pointwise regression.
- **Calibration** matters: the score ordering should track actual quality.
- **Reward hacking** is a real threat: verbosity, keyword stuffing, and circular repetition can fool a naively trained RM.
- **OOD generalisation** depends on training diversity — a narrowly trained RM fails on new domains.

> **Next chapter** → Chapter 10 introduces GRPO, RLOO, and KTO — modern algorithms that reduce reliance on an explicit reward model by using group-level comparisons or binary feedback.

In [ ]:
print('=== Reward Modeling — Final Summary ===')
print(f'Backbone              : {BACKBONE}')
print(f'Total parameters      : {n_params / 1e6:.1f} M')
print(f'Training pairs        : {len(TRAIN_PAIRS)}')
print(f'Validation pairs      : {len(VAL_PAIRS)}')
print(f'Final train accuracy  : {train_accs[-1]:.2%}')
print(f'Final val accuracy    : {val_accs[-1]:.2%}')
print(f'Calibration ρ         : {rho:.3f}')
print(f'OOD accuracy          : {ood_correct}/{len(OOD_PAIRS)} = {ood_correct/len(OOD_PAIRS):.2%}')
print(f'Reward hacking rate   : {n_hacked}/{len(ADVERSARIAL_EXAMPLES)} examples fooled')

## Part 8 -- Production Reward Models: Constitutional AI and Beyond

The reward model trained in this chapter is a lightweight binary preference classifier on top of Qwen2.5-0.5B.
Production reward models differ in three important ways:

### Anthropic's Constitutional AI (CAI)
Rather than relying solely on human raters, Anthropic's CAI pipeline uses **Claude itself** as a reward signal:
1. A draft response is generated.
2. Claude is prompted with a constitutional principle (e.g., *"Is this response harmful?"*) and asked to critique and revise.
3. The revised response is scored higher than the original -- creating synthetic preference pairs **at scale**.
4. A separate reward model is trained on these AI-generated pairs.

This removes the bottleneck of human annotation for the bulk of training data while keeping human feedback
at the top of the hierarchy for final calibration.

### Scale-up path
The table below shows how reward model capacity affects alignment quality:

| Reward model | Parameters | Typical use case |
|---|---|---|
| This notebook | 0.5 B + 2-layer head | Rapid prototyping, ablations |
| Qwen2.5-1.5B | 1.5 B + head | Small-scale RLHF experiments |
| Qwen2.5-7B | 7 B + head | Production preference modeling |
| Separate 6 B RM (InstructGPT) | 6 B | Full RLHF on 175 B policy |
| Claude-as-judge (CAI) | frontier scale | Constitutional AI pipelines |


In [ ]:
# Next scale up: Qwen2.5-1.5B as reward model backbone
# This cell shows the loading pattern -- no training is executed.
# Requires ~4 GB VRAM in float16.

from transformers import AutoModelForSequenceClassification, AutoTokenizer
# import torch  # already imported above

NEXT_SCALE_MODEL_ID = 'Qwen/Qwen2.5-1.5B'

print(f'To load {NEXT_SCALE_MODEL_ID} as a reward model backbone:')
print()
print('  tokenizer = AutoTokenizer.from_pretrained(NEXT_SCALE_MODEL_ID)')
print('  reward_model = AutoModelForSequenceClassification.from_pretrained(')
print(f'      "{NEXT_SCALE_MODEL_ID}",')
print('      num_labels=1,           # scalar reward output')
print('      torch_dtype=torch.float16,')
print('      device_map="auto",')
print('  )')
print()
print('Everything else -- Bradley-Terry loss, calibration, reward hacking -- stays identical.')
print('Only the backbone changes; the head and training loop are unchanged.')

# Uncomment below to actually load (requires ~4 GB VRAM):
# import torch
# tokenizer_15b = AutoTokenizer.from_pretrained(NEXT_SCALE_MODEL_ID)
# rm_15b = AutoModelForSequenceClassification.from_pretrained(
#     NEXT_SCALE_MODEL_ID,
#     num_labels=1,
#     torch_dtype=torch.float16,
#     device_map='auto',
# )
# print(f'Loaded: {sum(p.numel() for p in rm_15b.parameters())/1e9:.2f} B parameters')


## Summary -- Reward Modeling at Every Scale

**What we covered in Chapter 9:**
- Bradley-Terry preference model and its loss function
- Training a reward model from scratch on synthetic preference pairs
- Calibration analysis: does the score correlate with true quality?
- Reward hacking: how a policy can exploit reward model blind spots
- Out-of-distribution generalization and why it matters
- How Anthropic's Constitutional AI scales reward modeling via self-critique

**Looking ahead:** Chapter 10 explores **GRPO, RLOO, and KTO** -- three algorithms that bypass
explicit reward model training altogether. Instead of fitting a separate critic, they derive
optimization signals directly from group statistics or binary human feedback,
substantially reducing infrastructure complexity while remaining competitive with PPO+RM pipelines.
